# 05 — Agente de percepción-acción sobre video BDD100K
Orquesta ModelRegistry · AgentState · PerceptionEngine · OverlayRenderer · AlertLogger

In [1]:
# ── Celda 1 — Imports y rutas ──────────────────────────────────────────────
import sys, os, json, time
from pathlib import Path
import cv2
import numpy as np
from tqdm import tqdm

sys.path.insert(0, "src")  # notebook corre desde notebooks/

from model_manager import ModelRegistry
from agent_state   import AgentState
from perception    import PerceptionEngine
from overlay       import OverlayRenderer
from alert_logger   import AlertLogger
from risk_estimator import RiskEstimator
from ultralytics   import YOLO
import torch

BASE = Path("/mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks")
CKPT = BASE / "data/spatial_outputs"
OUT  = BASE / "data/spatial_outputs/video_outputs"
OUT.mkdir(parents=True, exist_ok=True)

print(f"CUDA disponible: {torch.cuda.is_available()}")
print(f"Outputs en: {OUT}")

/mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/env_ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA disponible: True
Outputs en: /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/video_outputs


In [2]:
# ── Celda 2 — Selección de video ───────────────────────────────────────────
BASE_BDDA = Path("/mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/videos/BDDA")

# Buscar videos de training — excluir 922.mp4 (corrupto)
videos = sorted([
    v for v in (BASE_BDDA / "training/camera_videos").glob("*.mp4")
    if v.stem != "922"
])

print(f"Videos disponibles: {len(videos)}")
for v in videos[:5]:
    print(f"  {v.name}")

# Seleccionar el primero
VIDEO_PATH = videos[0]
print(f"\nVideo seleccionado: {VIDEO_PATH.name}")

Videos disponibles: 925
  10.mp4
  1000.mp4
  1004.mp4
  1005.mp4
  1006.mp4

Video seleccionado: 10.mp4


In [3]:
# ── Celda 3 — Inicializar ModelRegistry con los 6 modelos ─────────────────
def loader_yolo(path):
    return YOLO(str(path))

def loader_torch(path):
    return torch.load(path, map_location="cpu", weights_only=False)

registry = ModelRegistry(vram_limit_gb=5.0)
registry.register_model("yolo",      BASE / "yolov8n.pt",                                     loader_yolo,  0.6)
registry.register_model("segformer", CKPT / "segformer_best.pt",                              loader_torch, 1.2)
registry.register_model("resnet",    CKPT / "resnet_best.pt",                                 loader_torch, 0.3)
registry.register_model("gcn",       CKPT / "checkpoints_03/gcn_best.pt",                     loader_torch, 0.1)
registry.register_model("lstm",      CKPT / "checkpoints_03/lstm_best.pt",                    loader_torch, 0.1)
registry.register_model("bnn",       CKPT / "prediction_outputs/checkpoints_bnn/bnn_best.pt", loader_torch, 0.2)
registry.status()

[ModelRegistry] Dispositivo : cuda
[ModelRegistry] VRAM libre  : 5.65 GB / 5.80 GB
[ModelRegistry] Límite fijado: 5.0 GB
[ModelRegistry] Registrado  : 'yolo' | VRAM estimada: 0.6 GB | checkpoint: yolov8n.pt
[ModelRegistry] Registrado  : 'segformer' | VRAM estimada: 1.2 GB | checkpoint: segformer_best.pt
[ModelRegistry] Registrado  : 'resnet' | VRAM estimada: 0.3 GB | checkpoint: resnet_best.pt
[ModelRegistry] Registrado  : 'gcn' | VRAM estimada: 0.1 GB | checkpoint: gcn_best.pt
[ModelRegistry] Registrado  : 'lstm' | VRAM estimada: 0.1 GB | checkpoint: lstm_best.pt
[ModelRegistry] Registrado  : 'bnn' | VRAM estimada: 0.2 GB | checkpoint: bnn_best.pt

[ModelRegistry] Estado — VRAM en uso: 0.0 GB / límite: 5.0 GB
  Nombre         Cargado    En GPU   VRAM (GB)  Checkpoint
  -------------- ---------- -------- ---------- ------------------------------
  yolo           no         no       0.6        yolov8n.pt
  segformer      no         no       1.2        segformer_best.pt
  resnet         

In [4]:
# ── Celda 4 — Inicializar agente y componentes ─────────────────────────────
agent    = AgentState()
perceive = PerceptionEngine(registry)
renderer = OverlayRenderer(alpha_seg=0.45, alpha_hud=0.6)
logger   = AlertLogger(
    csv_path        = OUT / "video_alerts.csv",
    json_path       = OUT / "video_metrics.json",
    debounce_frames = 10,
)
risk_est = RiskEstimator(registry, seq_len=10)
print("[\u2713] Agente y componentes inicializados.")

[✓] Agente y componentes inicializados.


In [5]:
# ── Celda 5 — Inspeccionar video de entrada ────────────────────────────────
cap = cv2.VideoCapture(str(VIDEO_PATH))
fps_src  = cap.get(cv2.CAP_PROP_FPS)
width    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
cap.release()

print(f"Resolución  : {width}\u00d7{height}")
print(f"FPS fuente  : {fps_src:.1f}")
print(f"Frames total: {n_frames}")
print(f"Duración    : {n_frames/fps_src:.1f} s")

Resolución  : 1280×720
FPS fuente  : 59.9
Frames total: 600
Duración    : 10.0 s


In [6]:
# ── Celda 6 — Loop principal del agente ───────────────────────────────────
CLAHE_MODE  = "normal"   # "normal" | "night" | "weather"
MAX_FRAMES  = None       # None = todo el video, int = limitar
FRAME_SKIP  = 1          # procesar 1 de cada N frames

cap    = cv2.VideoCapture(str(VIDEO_PATH))
writer = cv2.VideoWriter(
    str(OUT / "video_output.mp4"),
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps_src,
    (width, height),
)

t_inicio    = time.time()
latencias   = []
frame_idx   = 0
frames_proc = 0

try:
    pbar = tqdm(total=n_frames, desc="Agente", unit="fr")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        if MAX_FRAMES and frame_idx >= MAX_FRAMES:
            break

        frame_idx += 1
        if frame_idx % FRAME_SKIP != 0:
            pbar.update(1)
            continue

        t0 = time.time()

        # 1. Percepción
        active     = agent.active_models()
        percep_out = perceive.run(frame, active, CLAHE_MODE)

        # 2. Estimar riesgo real con GCN+LSTM+BNN (siempre, todos los frames)
        risk_score, sigma = risk_est.estimate(
            percep_out["boxes"],
            frame.shape,
        )

        level = agent.update(frame, risk_score)

        # 3. Overlay visual
        frame_out = renderer.render(
            percep_out["frame_clahe"],
            percep_out,
            level,
            frames_proc,
            uncertainty=sigma if sigma > 0 else None,
        )

        # 4. Resize al tamaño original si difiere
        if frame_out.shape[:2] != (height, width):
            frame_out = cv2.resize(frame_out, (width, height))

        # 5. Escribir frame anotado
        writer.write(frame_out)

        # 6. Logger
        lat_ms = (time.time() - t0) * 1000
        latencias.append(lat_ms)
        logger.update(frames_proc, level, risk_score,
                      uncertainty=sigma if sigma > 0 else None,
                      timestamp_ms=lat_ms)

        frames_proc += 1
        pbar.set_postfix(
            level=level,
            risk=f"{risk_score:.3f}",
            sigma=f"{sigma:.3f}",
            lat=f"{lat_ms:.0f}ms",
        )
        pbar.update(1)

finally:
    pbar.close()
    cap.release()
    writer.release()

t_total  = time.time() - t_inicio
fps_real = frames_proc / t_total if t_total > 0 else 0

print(f"\nFrames procesados : {frames_proc}")
print(f"Tiempo total      : {t_total:.1f} s")
print(f"FPS real          : {fps_real:.1f}")
print(f"Latencia media    : {np.mean(latencias):.1f} ms")

Agente:   0%|                                                                                                    | 0/600 [00:00<?, ?fr/s]

[ModelRegistry] Cargando 'yolo' desde disco ...
[ModelRegistry] VRAM (tras cargar 'yolo'): PyTorch=0.01 GB | libre=5.62 GB / 5.80 GB


Agente:   1%|▎                                             | 4/600 [00:01<11:11,  1.13s/fr, lat=22ms, level=low, risk=0.500, sigma=0.000]

[ModelRegistry] Cargando 'gcn' desde disco ...
[ModelRegistry] VRAM (tras cargar 'gcn'): PyTorch=0.04 GB | libre=5.54 GB / 5.80 GB


Agente:   2%|▋                                          | 9/600 [00:01<00:59,  9.89fr/s, lat=77ms, level=medium, risk=0.673, sigma=0.021]

[ModelRegistry] Cargando 'lstm' desde disco ...
[ModelRegistry] VRAM (tras cargar 'lstm'): PyTorch=0.05 GB | libre=5.54 GB / 5.80 GB
[ModelRegistry] Cargando 'bnn' desde disco ...
[ModelRegistry] VRAM (tras cargar 'bnn'): PyTorch=0.05 GB | libre=5.52 GB / 5.80 GB
[ModelRegistry] Cargando 'segformer' desde disco ...
[ModelRegistry] VRAM (tras cargar 'segformer'): PyTorch=0.05 GB | libre=5.52 GB / 5.80 GB



Agente:   2%|▉                                           | 12/600 [00:03<02:58,  3.30fr/s, lat=80ms, level=high, risk=0.682, sigma=0.016]

[ModelRegistry] Cargando 'resnet' desde disco ...
[ModelRegistry] VRAM (tras cargar 'resnet'): PyTorch=0.13 GB | libre=5.30 GB / 5.80 GB


Agente: 100%|██████████████████████████████████████████| 600/600 [01:15<00:00,  7.97fr/s, lat=126ms, level=high, risk=0.754, sigma=0.024]


Frames procesados : 600
Tiempo total      : 75.3 s
FPS real          : 8.0
Latencia media    : 122.3 ms


In [7]:
# ── Celda 6b — Generar video_seg.mp4 (solo segmentación semántica) ────────
ALERT_COLOR = {"low": (0, 255, 0), "medium": (0, 165, 255), "high": (0, 0, 255)}

def make_writer(name):
    return cv2.VideoWriter(
        str(OUT / name),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps_src,
        (width, height),
    )

# Reiniciar agente y estimador para replay limpio
agent2    = AgentState()
risk_est2 = RiskEstimator(registry, seq_len=10)

cap2       = cv2.VideoCapture(str(VIDEO_PATH))
writer_seg = make_writer("video_seg.mp4")

frame_idx2   = 0
frames_proc2 = 0

try:
    pbar2 = tqdm(total=n_frames, desc="Seg-only", unit="fr")
    while True:
        ret, frame = cap2.read()
        if not ret:
            break
        frame_idx2 += 1

        active     = agent2.active_models()
        percep_out = perceive.run(frame, active, CLAHE_MODE)
        risk_score, sigma = risk_est2.estimate(percep_out["boxes"], frame.shape)
        agent2.update(frame, risk_score)

        frame_seg = percep_out["frame_clahe"].copy()
        if percep_out.get("seg_mask") is not None:
            frame_seg = renderer._draw_seg_mask(frame_seg, percep_out["seg_mask"])
        if frame_seg.shape[:2] != (height, width):
            frame_seg = cv2.resize(frame_seg, (width, height))
        writer_seg.write(frame_seg)

        frames_proc2 += 1
        pbar2.update(1)
finally:
    pbar2.close()
    cap2.release()
    writer_seg.release()

print(f"[✓] video_seg.mp4 generado — {frames_proc2} frames")


Seg-only: 100%|████████████████████████████████████████████████████████████████████████████████████████| 600/600 [00:56<00:00, 10.71fr/s]

[✓] video_seg.mp4 generado — 600 frames


In [8]:
# ── Celda 6c — Generar video_alerts.mp4 (solo texto de alerta centrado) ───
agent3    = AgentState()
risk_est3 = RiskEstimator(registry, seq_len=10)

cap3          = cv2.VideoCapture(str(VIDEO_PATH))
writer_alerts = make_writer("video_alerts.mp4")

frame_idx3   = 0
frames_proc3 = 0

try:
    pbar3 = tqdm(total=n_frames, desc="Alerts-only", unit="fr")
    while True:
        ret, frame = cap3.read()
        if not ret:
            break
        frame_idx3 += 1

        active     = agent3.active_models()
        percep_out = perceive.run(frame, active, CLAHE_MODE)
        risk_score, sigma = risk_est3.estimate(percep_out["boxes"], frame.shape)
        level = agent3.update(frame, risk_score)

        frame_alert = percep_out["frame_clahe"].copy()
        if frame_alert.shape[:2] != (height, width):
            frame_alert = cv2.resize(frame_alert, (width, height))

        color = ALERT_COLOR.get(level, (255, 255, 255))
        label = f"NIVEL: {level.upper()}  |  RIESGO: {risk_score:.3f}"
        if sigma > 0:
            label += f"  |  sigma: {sigma:.3f}"

        overlay_a = frame_alert.copy()
        tw, th = 900, 60
        tx, ty = (width - tw) // 2, (height - th) // 2
        cv2.rectangle(overlay_a, (tx, ty), (tx + tw, ty + th), (0, 0, 0), -1)
        cv2.addWeighted(overlay_a, 0.55, frame_alert, 0.45, 0, frame_alert)
        cv2.putText(frame_alert, label,
                    (tx + 20, ty + 40),
                    cv2.FONT_HERSHEY_DUPLEX, 0.85, color, 2, cv2.LINE_AA)
        writer_alerts.write(frame_alert)

        frames_proc3 += 1
        pbar3.update(1)
finally:
    pbar3.close()
    cap3.release()
    writer_alerts.release()

print(f"[✓] video_alerts.mp4 generado — {frames_proc3} frames")


Alerts-only: 100%|█████████████████████████████████████████████████████████████████████████████████████| 600/600 [01:06<00:00,  9.05fr/s]

[✓] video_alerts.mp4 generado — 600 frames


In [9]:
# ── Celda 7 — Guardar outputs finales ─────────────────────────────────────
logger.save_metrics(
    video_path=str(VIDEO_PATH),
    fps_real=fps_real,
)
logger.summary()

print(f"\nOutputs guardados en {OUT}:")
for f in sorted(OUT.glob("video_*")):
    size_mb = f.stat().st_size / 1024**2
    print(f"  {f.name}  ({size_mb:.1f} MB)")

[AlertLogger] Métricas guardadas → /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/video_outputs/video_metrics.json

[AlertLogger] ── Resumen de sesión ──────────────────────
  Total frames procesados : 600
  Frames low              :     9  (1.5%)
  Frames medium           :     2  (0.3%)
  Frames high             :   589  (98.2%)
  Alertas medium          : 1
  Alertas high            : 59
  Latencia media          : 122.34 ms
[AlertLogger] ────────────────────────────────────────────

Outputs guardados en /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/video_outputs:
  video_alerts.csv  (0.0 MB)
  video_alerts.mp4  (13.9 MB)
  video_metrics.json  (0.0 MB)
  video_output.mp4  (15.0 MB)
  video_seg.mp4  (12.9 MB)


In [10]:
# ── Celda 8 — Offload final y estado VRAM ─────────────────────────────────
registry.offload_all()
registry.status()

[ModelRegistry] Offloading 'bnn' GPU → CPU ...
[ModelRegistry] VRAM (tras offload 'bnn'): PyTorch=0.11 GB | libre=5.46 GB / 5.80 GB
[ModelRegistry] Offloading 'resnet' GPU → CPU ...
[ModelRegistry] VRAM (tras offload 'resnet'): PyTorch=0.06 GB | libre=5.53 GB / 5.80 GB
[ModelRegistry] Offloading 'segformer' GPU → CPU ...
[ModelRegistry] VRAM (tras offload 'segformer'): PyTorch=0.05 GB | libre=5.54 GB / 5.80 GB
[ModelRegistry] Offloading 'lstm' GPU → CPU ...
[ModelRegistry] VRAM (tras offload 'lstm'): PyTorch=0.05 GB | libre=5.54 GB / 5.80 GB
[ModelRegistry] Offloading 'yolo' GPU → CPU ...
[ModelRegistry] VRAM (tras offload 'yolo'): PyTorch=0.04 GB | libre=5.54 GB / 5.80 GB
[ModelRegistry] Offloading 'gcn' GPU → CPU ...
[ModelRegistry] VRAM (tras offload 'gcn'): PyTorch=0.03 GB | libre=5.57 GB / 5.80 GB
[ModelRegistry] Todos los modelos offloadeados.

[ModelRegistry] Estado — VRAM en uso: 0.0 GB / límite: 5.0 GB
  Nombre         Cargado    En GPU   VRAM (GB)  Checkpoint
  --------------

In [12]:
# ── Celda 9 — 3 videos por cada .mov en videosFinales (formato visual mejorado) ──
VIDEOS_DIR  = Path("/mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/videosFinales")
OUT_FINALES = Path("/mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/video_outputs/finales")
OUT_FINALES.mkdir(parents=True, exist_ok=True)

CLAHE_MODE_F = "normal"

# Colores BGR por nivel (verde=bajo, amarillo=medio, rojo=alto)
NIVEL_COLOR = {"low": (0, 255, 0), "medium": (0, 255, 255), "high": (0, 0, 255)}
NIVEL_TEXT  = {"low": "RIESGO BAJO", "medium": "PRECAUCION", "high": "PELIGRO ALTO"}

def make_writer_f(path, fps, w, h):
    return cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))

def draw_alert_boxes(frame, boxes_result, level):
    """Dibuja boxes YOLO coloreados por nivel — recibe Results object de ultralytics."""
    if boxes_result is None or not hasattr(boxes_result, "boxes") or boxes_result.boxes is None:
        return frame, 0
    det = boxes_result.boxes
    if len(det) == 0:
        return frame, 0
    color = NIVEL_COLOR.get(level, (0, 255, 0))
    texto = NIVEL_TEXT.get(level, "")
    xyxy_all = det.xyxy.cpu().numpy().astype(int)
    conf_all = det.conf.cpu().numpy()
    for (x1, y1, x2, y2), conf in zip(xyxy_all, conf_all):
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
        label = f"{texto} {conf*100:.0f}%"
        (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(frame, (x1, y1 - 20), (x1 + tw, y1), color, -1)
        cv2.putText(frame, label, (x1, y1 - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 0, 0), 1)
    return frame, len(det)

movs = sorted(VIDEOS_DIR.glob("*.mov"))
print(f"Videos encontrados: {[v.name for v in movs]}\n")

for VIDEO in movs:
    stem = VIDEO.stem
    print(f"\n{'='*60}\nProcesando: {VIDEO.name}")

    cap_info = cv2.VideoCapture(str(VIDEO))
    fps_v = cap_info.get(cv2.CAP_PROP_FPS) or 30.0
    w_v   = int(cap_info.get(cv2.CAP_PROP_FRAME_WIDTH))
    h_v   = int(cap_info.get(cv2.CAP_PROP_FRAME_HEIGHT))
    nf_v  = int(cap_info.get(cv2.CAP_PROP_FRAME_COUNT))
    cap_info.release()
    print(f"  {w_v}x{h_v} | {fps_v:.1f} FPS | {nf_v} frames")

    w_full   = make_writer_f(OUT_FINALES / f"{stem}_full.mp4",   fps_v, w_v, h_v)
    w_seg    = make_writer_f(OUT_FINALES / f"{stem}_seg.mp4",    fps_v, w_v, h_v)
    w_alerts = make_writer_f(OUT_FINALES / f"{stem}_alerts.mp4", fps_v, w_v, h_v)

    # CSV de alertas
    csv_path = OUT_FINALES / f"{stem}_registro_alertas.csv"
    csv_file = open(csv_path, mode="w", newline="", encoding="utf-8")
    import csv as csv_mod
    escritor = csv_mod.writer(csv_file)
    escritor.writerow(["Frame", "Tiempo (Segundos)", "Risk_Score", "Sigma", "Nivel", "Num_Objetos"])

    agent_f    = AgentState()
    risk_est_f = RiskEstimator(registry, seq_len=10)
    cap_f      = cv2.VideoCapture(str(VIDEO))
    latencias_f = []
    frames_f    = 0
    t0_total    = time.time()

    try:
        pbar_f = tqdm(total=nf_v, desc=stem, unit="fr")
        while True:
            ret, frame = cap_f.read()
            if not ret:
                break

            active     = agent_f.active_models()
            percep_out = perceive.run(frame, active, CLAHE_MODE_F)
            risk_score, sigma = risk_est_f.estimate(percep_out["boxes"], frame.shape)
            level = agent_f.update(frame, risk_score)

            t1   = time.time()
            base = percep_out["frame_clahe"].copy()
            boxes_result = percep_out.get("boxes")

            # ── Video full: boxes coloreados + HUD BNN ────────────────────
            frame_full = base.copy()
            frame_full, n_det = draw_alert_boxes(frame_full, boxes_result, level)
            hud_label = f"Risk:{risk_score:.3f} | {NIVEL_TEXT.get(level,'')} | sigma:{sigma:.3f}"
            cv2.rectangle(frame_full, (0, 0), (520, 32), (0, 0, 0), -1)
            cv2.putText(frame_full, hud_label, (8, 22),
                        cv2.FONT_HERSHEY_DUPLEX, 0.65,
                        NIVEL_COLOR.get(level, (255, 255, 255)), 1, cv2.LINE_AA)
            if frame_full.shape[:2] != (h_v, w_v):
                frame_full = cv2.resize(frame_full, (w_v, h_v))
            w_full.write(frame_full)

            # ── Video seg: máscara SegFormer ──────────────────────────────
            frame_seg = base.copy()
            if percep_out.get("mask") is not None:
                frame_seg = renderer._draw_seg_mask(frame_seg, percep_out["mask"])
            if frame_seg.shape[:2] != (h_v, w_v):
                frame_seg = cv2.resize(frame_seg, (w_v, h_v))
            w_seg.write(frame_seg)

            # ── Video alerts: boxes + texto ───────────────────────────────
            frame_alert = base.copy()
            frame_alert, _ = draw_alert_boxes(frame_alert, boxes_result, level)
            if frame_alert.shape[:2] != (h_v, w_v):
                frame_alert = cv2.resize(frame_alert, (w_v, h_v))
            w_alerts.write(frame_alert)

            # CSV — registrar si nivel medio o alto
            tiempo_s = round(frames_f / fps_v, 2)
            if level in ("medium", "high"):
                escritor.writerow([frames_f, tiempo_s, round(risk_score, 4),
                                   round(sigma, 4), level, n_det])

            lat_ms = (time.time() - t1) * 1000
            latencias_f.append(lat_ms)
            frames_f += 1
            pbar_f.set_postfix(level=level, risk=f"{risk_score:.3f}", lat=f"{lat_ms:.0f}ms")
            pbar_f.update(1)

    finally:
        pbar_f.close()
        cap_f.release()
        csv_file.close()
        w_full.release()
        w_seg.release()
        w_alerts.release()

    t_tot = time.time() - t0_total
    print(f"  [✓] {frames_f} frames | {frames_f/t_tot:.1f} FPS real | lat media {np.mean(latencias_f):.0f} ms")
    print(f"  Outputs: {stem}_full.mp4 | {stem}_seg.mp4 | {stem}_alerts.mp4 | {stem}_registro_alertas.csv")

print(f"\n[✓] Todos los videos procesados → {OUT_FINALES}")


Videos encontrados: ['DiaPeatones.mov', 'MedioDiaLluvia.mov', 'dia1.mov']


Procesando: DiaPeatones.mov
  1280x720 | 30.0 FPS | 604 frames


DiaPeatones:   2%|▊                                                   | 10/604 [00:00<00:36, 16.41fr/s, lat=30ms, level=high, risk=0.693]

[ModelRegistry] Moviendo 'lstm' CPU → GPU ...
[ModelRegistry] VRAM (tras mover 'lstm' a GPU): PyTorch=0.05 GB | libre=5.53 GB / 5.80 GB
[ModelRegistry] Moviendo 'bnn' CPU → GPU ...
[ModelRegistry] VRAM (tras mover 'bnn' a GPU): PyTorch=0.05 GB | libre=5.51 GB / 5.80 GB
[ModelRegistry] Moviendo 'segformer' CPU → GPU ...
[ModelRegistry] VRAM (tras mover 'segformer' a GPU): PyTorch=0.06 GB | libre=5.51 GB / 5.80 GB
[ModelRegistry] Moviendo 'resnet' CPU → GPU ...
[ModelRegistry] VRAM (tras mover 'resnet' a GPU): PyTorch=0.18 GB | libre=5.31 GB / 5.80 GB


DiaPeatones: 100%|███████████████████████████████████████████████████| 604/604 [01:45<00:00,  5.72fr/s, lat=67ms, level=high, risk=0.702]


  [✓] 604 frames | 5.7 FPS real | lat media 67 ms
  Outputs: DiaPeatones_full.mp4 | DiaPeatones_seg.mp4 | DiaPeatones_alerts.mp4 | DiaPeatones_registro_alertas.csv

Procesando: MedioDiaLluvia.mov
  1280x720 | 30.1 FPS | 1203 frames


MedioDiaLluvia: 100%|██████████████████████████████████████████████| 1203/1203 [03:26<00:00,  5.82fr/s, lat=75ms, level=high, risk=0.682]


  [✓] 1203 frames | 5.8 FPS real | lat media 67 ms
  Outputs: MedioDiaLluvia_full.mp4 | MedioDiaLluvia_seg.mp4 | MedioDiaLluvia_alerts.mp4 | MedioDiaLluvia_registro_alertas.csv

Procesando: dia1.mov
  1280x720 | 30.0 FPS | 1208 frames


dia1: 100%|████████████████████████████████████████████████████████| 1208/1208 [03:30<00:00,  5.74fr/s, lat=59ms, level=high, risk=0.720]

  [✓] 1208 frames | 5.7 FPS real | lat media 67 ms
  Outputs: dia1_full.mp4 | dia1_seg.mp4 | dia1_alerts.mp4 | dia1_registro_alertas.csv

[✓] Todos los videos procesados → /mnt/bigdata/pipeline_samsung/Equipo8-Grupo8-SIC-2025-/notebooks/data/spatial_outputs/video_outputs/finales
